In [9]:
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine('sqlite:///movies.db')

from sqlalchemy import text


min_year = 2000

query = text("""
SELECT 
    m.title,
    CAST(SUBSTR(m.title, LENGTH(m.title) - 4, 4) AS INTEGER) AS year,
    CAST(AVG(r.rating) AS INT) AS avg_rating_int
FROM movies m
JOIN ratings r ON m.movieId = r.movieId
WHERE CAST(SUBSTR(m.title, LENGTH(m.title) - 4, 4) AS INTEGER) >= :min_year
GROUP BY m.title
ORDER BY RANDOM()
LIMIT 10
""")

results = pd.read_sql_query(query, engine, params={'min_year': min_year})
results


,title,year,avg_rating_int
0,A.I. Artificial Intelligence (2001),2001,3
1,Captain America: Civil War (2016),2016,3
2,Intouchables (2011),2011,4
3,Sex Drive (2008),2008,3
4,Bad Santa (2003),2003,3
5,"Kings of Summer, The (2013)",2013,3
6,Shadow World (2016),2016,4
7,Crimson Peak (2015),2015,3
8,"Equalizer, The (2014)",2014,3
9,Catch and Release (2006),2006,2


In [10]:
query = text("""
SELECT m.title, AVG(r.rating) as avg_rating
FROM movies m
JOIN ratings r ON m.movieId = r.movieId
GROUP BY m.title
ORDER BY avg_rating DESC
LIMIT 10
""")

top_movies = pd.read_sql_query(query, engine)
top_movies.head()

,title,avg_rating
0,Zeitgeist: Moving Forward (2011),5.0
1,Wow! A Talking Fish! (1983),5.0
2,World of Glory (1991),5.0
3,Wonder Woman (2009),5.0
4,Won't You Be My Neighbor? (2018),5.0


In [12]:

query = text("""
SELECT 
    m.title,
    t.tag,
    CAST(AVG(r.rating) AS INT) AS avg_rating_int
FROM movies m
JOIN ratings r ON m.movieId = r.movieId
JOIN tags t ON m.movieId = t.movieId
GROUP BY m.title, t.tag
ORDER BY RANDOM()
LIMIT 30
""")

movie_tag_ratings = pd.read_sql_query(query, engine)
movie_tag_ratings


,title,tag,avg_rating_int
0,"Pianist, The (2002)",holocaust,4
1,Corpse Bride (2005),visually appealing,3
2,Monty Python and the Holy Grail (1975),England,4
3,Dead Again (1991),memory,3
4,Who Killed Chea Vichea? (2010),procedural,5
5,X-Men: The Last Stand (2006),Hugh Jackman,3
6,Hero (Ying xiong) (2002),martial arts,3
7,Buffalo '66 (a.k.a. Buffalo 66) (1998),avant-garde romantic comedy,3
8,There Will Be Blood (2007),morality,4
9,It Could Happen to You (1994),gambling,3


In [16]:
query = text("""
SELECT
    m.movieID,
    m.title,
    COUNT(r.rating) AS num_ratings
FROM movies m
JOIN ratings r ON m.movieId = r.movieId
JOIN tags t ON m.movieId = t.movieId
GROUP BY m.title, m.movieID
ORDER BY num_ratings DESC
LIMIT 30
""")

top_rated_movies = pd.read_sql_query(query, engine)
top_rated_movies


,movieId,title,num_ratings
0,296,Pulp Fiction (1994),55567
1,2959,Fight Club (1999),11772
2,260,Star Wars: Episode IV - A New Hope (1977),6526
3,293,Léon: The Professional (a.k.a. The Professiona...,4655
4,924,2001: A Space Odyssey (1968),4469
5,7361,Eternal Sunshine of the Spotless Mind (2004),4454
6,79132,Inception (2010),3718
7,1732,"Big Lebowski, The (1998)",3392
8,4878,Donnie Darko (2001),3161
9,356,Forrest Gump (1994),2961


In [21]:
from sqlalchemy import text
import pandas as pd

query = text("""
SELECT 
    m.title,
    t.tag,
    COUNT(*) AS tag_count,
    AVG(r.rating) AS avg_rating,
    COUNT(r.rating) AS num_ratings
FROM movies m
JOIN ratings r ON m.movieID = r.movieID
JOIN tags t ON m.movieID = t.movieID
WHERE m.movieID IN (
    -- Subquery: to 20 most rated movies
    SELECT m.movieId
    FROM movies m
    JOIN ratings r ON m.movieId = r.movieId
    GROUP BY m.movieId
    ORDER BY COUNT(r.rating) DESC
    LIMIT 20
) 
GROUP BY m.title, t.tag
ORDER BY num_ratings DESC, tag_count DESC
LIMIT 50
""")

# Run the query
common_tags = pd.read_sql_query(query, engine)

# Display the results
common_tags


,title,tag,tag_count,avg_rating,num_ratings
0,Star Wars: Episode IV - A New Hope (1977),sci-fi,753,4.231076,753
1,Star Wars: Episode IV - A New Hope (1977),classic sci-fi,753,4.231076,753
2,Fight Club (1999),dark comedy,654,4.272936,654
3,Pulp Fiction (1994),non-linear,614,4.197068,614
4,Pulp Fiction (1994),hit men,614,4.197068,614
5,Pulp Fiction (1994),great soundtrack,614,4.197068,614
6,Pulp Fiction (1994),good dialogue,614,4.197068,614
7,Pulp Fiction (1994),drugs,614,4.197068,614
8,Pulp Fiction (1994),cult film,614,4.197068,614
9,Pulp Fiction (1994),Tarantino,614,4.197068,614
